# Instrument profiles

Broaden synthetic spectra to nominal instrument resolving powers and select detector coverage. The profiles work offline. The examples include source padding for convolution. Throughput and detector noise require separate modeling.

In [ ]:
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from speclib import Spectrum, get_profile, list_instrument_profiles

wave = np.geomspace(0.5, 6.0, 30000) * u.um
x = wave.to_value(u.um)
flux = np.ones(wave.size)
for center in (1.2, 3.5, 4.5):
    flux += 4 * np.exp(-0.5 * ((x - center) / 0.0004)**2)
unit = u.erg / (u.s * u.cm**2 * u.AA)
source = Spectrum(spectral_axis=wave, flux=flux * unit)
list_instrument_profiles()


## Coverage and sampling

The default retains source samples within `valid_intervals`. Automatic sampling creates wavelength centers at a chosen number of samples per resolution element. Use explicit wavelengths or bins for detector sampling. Resampling integrates bins; coarse bins and sparse samples can increase apparent widths. Use finer sampling for width measurements.

In [ ]:
profile = get_profile("JWST/NIRSpec/PRISM/CLEAR")
print(profile.configuration)
print(profile.valid_intervals)
print(profile.calibration_version)
print(profile.resolution_definition)
print(profile.assumptions)
print(profile.provenance["url"])  # Original FITS, independent of coverage source.
print(profile.provenance["sha256"])
print(profile.coverage_source_url)
assert profile.provenance["filename"] == "jwst_nirspec_prism_disp.fits"

prism = source.observed_by(profile)
sampled = source.observed_by(profile, samples_per_resolution_element=3)
explicit_centers = np.linspace(0.65, 5.2, 300) * u.um
explicit = source.observed_by(profile, wavelength=explicit_centers)
assert len(prism.flux) > len(sampled.flux)
assert np.array_equal(explicit.wavelength, explicit_centers)
print(prism.valid_intervals)
continuous = prism


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(profile.wavelength, profile.resolving_power)
axes[0].set(xlabel="Wavelength (micrometers)", ylabel="Nominal resolving power", title="NIRSpec PRISM/CLEAR")
axes[1].plot(source.wavelength.to_value(u.um), source.flux.value, label="Source")
axes[1].plot(continuous.wavelength.to_value(u.um), continuous.flux.value, label="Broadened")
axes[1].plot(sampled.wavelength.to_value(u.um), sampled.flux.value, ".", label="3 samples/FWHM")
axes[1].set(xlim=(1.15, 1.25), xlabel="Wavelength (micrometers)", ylabel="Flux density (arbitrary units)")
axes[1].legend()
fig.tight_layout()
plt.show()


## Detector gaps

G395H/F290LP has a 3.72–3.82 µm gap in approximate BOTS/S1600A1/SUB2048 coverage. Source light within the output gap still contributes to convolution. Plot intervals separately; resampling and binning reject bins that cross a gap.

For a preloaded high-resolution grid, use `grid.observed_by(...)`, then `get_spectrum(...)` to retain detector coverage and provenance. See the [spectral-grids tutorial](spectral_grids.ipynb).

In [ ]:
gapped = source.observed_by("jwst/nirspec/g395h/f290lp")
print(gapped.valid_intervals)
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
for lower, upper in gapped.valid_intervals:
    selected = (gapped.spectral_axis >= lower) & (gapped.spectral_axis <= upper)
    ax.plot(gapped.spectral_axis[selected].to_value(u.um), gapped.flux[selected].value)
ax.axvspan(3.72, 3.82, color="0.9", label="Detector gap")
ax.set(xlabel="Wavelength (micrometers)", ylabel="Flux density (arbitrary units)")
ax.legend()
plt.show()
assert not np.any((gapped.wavelength > 3.72*u.um) & (gapped.wavelength < 3.82*u.um))
centers = np.r_[np.linspace(3.0, 3.6, 20), np.linspace(4., 5., 20)]*u.um
resampled = gapped.resample(centers)
assert len(resampled.valid_intervals) == 2
selected = gapped.select_wavelength(4.*u.um, 5.*u.um)
assert len(selected.valid_intervals) == 1
try:
    gapped.resample([3.76, 3.77, 3.78]*u.um)
except ValueError as error:
    print(error)
else:
    raise AssertionError("Gap request was not rejected")
assert source.wavelength.min() == 0.5*u.um
assert "instrument_profile" not in source.meta


## Select a NIRSpec subarray

Short identifiers select SUB2048. Other regular subarrays use `jwst/nirspec/bots/s1600a1/<subarray>/<disperser>/<filter>`. SUB1024A selects outer detector portions; SUB1024B selects inner portions. SUB512/SUB512S share spectral coverage but differ in spatial height. All reuse the nominal disperser R curve. PRISM/CLEAR with SUB1024A and multistripe subarrays are unavailable.

In [ ]:
subarray_profile = get_profile("jwst/nirspec/bots/s1600a1/sub1024a/g395h/f290lp")
subarray_result = source.observed_by(subarray_profile)
print(subarray_profile.configuration, subarray_result.valid_intervals)
np.testing.assert_array_equal(subarray_result.valid_intervals.to_value(u.um),
                              [[2.87, 3.03], [4.51, 5.18]])
# The omitted middle region includes unread columns as well as the detector gap.
print("Physical detector gap:", subarray_profile.provenance["nominal_detector_gap_um"])
assert not np.any((subarray_result.wavelength > 3.03*u.um) &
                  (subarray_result.wavelength < 4.51*u.um))


## STIS, WFC3, and SOSS orders

STIS assumes a centered point source through 52X2 at the prime CCD setting. WFC3 uses nominal constant wavelength FWHM. SOSS assumes CLEAR/SUBSTRIP256 and treats each order independently, without decontamination or stitching. Order 2 excludes wavelengths above 1 µm because of low signal and uncertain photometric calibration. The [instrument reference](../user_guide/instrument_provenance.rst) gives sources and equations.

In [ ]:
optical_wave = np.geomspace(0.25, 3.0, 40000) * u.um
optical_source = Spectrum(spectral_axis=optical_wave,
                          flux=np.ones(optical_wave.size) * source.flux.unit)
new_modes = ["hst/stis/g430l", "hst/stis/g750l", "hst/wfc3/g102",
             "hst/wfc3/g141", "jwst/niriss/soss/order1", "jwst/niriss/soss/order2"]
matched = {}
for mode in new_modes:
    calibration = get_profile(mode)
    matched[mode] = optical_source.observed_by(calibration)
    assert type(matched[mode]) is Spectrum
    print(mode, calibration.quality, matched[mode].valid_intervals)
    print(calibration.provenance["equation"])

fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
for mode in new_modes:
    calibration = get_profile(mode)
    plot_wave = np.linspace(*calibration.valid_intervals[0].to_value(u.um), 300) * u.um
    ax.plot(plot_wave, calibration.resolving_power_at(plot_wave), label=mode)
ax.set(xlabel="Wavelength (micrometers)", ylabel="Nominal resolving power")
ax.legend(fontsize="small")


## NIRCam Module A Grism R

The four filters share the nominal first-order point-source prediction of [Greene et al. (2017)](https://doi.org/10.1117/1.JATIS.3.3.035001), valid on 2.4–5.0 µm. Output is trimmed to 2.410 µm for F277W and 4.985 µm for F444W to retain convolution padding; F322W2/F356W keep their 5% filter limits. Faint second orders can affect F277W/F322W2. Grism C, Module B, and wide-field modes are unsupported.

In [ ]:
nircam_wave = np.geomspace(2.39, 5.01, 24000) * u.um
nircam_flux = np.ones(nircam_wave.size)
for center in (2.7, 3.3, 3.6, 4.5):
    nircam_flux -= 0.7 * np.exp(-0.5 * ((nircam_wave.value-center)/0.00012)**2)
nircam_source = Spectrum(spectral_axis=nircam_wave, flux=nircam_flux*source.flux.unit)
fig, axes = plt.subplots(2, 2, figsize=(10, 6), layout="constrained")
for name, ax in zip(("f277w", "f322w2", "f356w", "f444w"), axes.flat):
    mode = "jwst/nircam/grismr/" + name
    result = nircam_source.observed_by(mode)  # Preserve native sampling.
    print(get_profile(mode).mode, result.valid_intervals)
    ax.plot(nircam_source.wavelength.to_value(u.um), nircam_source.flux.value,
            alpha=0.35, label="Input")
    ax.plot(result.wavelength.to_value(u.um), result.flux.value, label="Matched")
    ax.set(title=name.upper(), xlim=result.valid_intervals[0].to_value(u.um),
           xlabel="Wavelength (µm)", ylabel="Relative flux")
    ax.legend()

# Optional sampling uses the same API as every other profile.
f444w_sampled = nircam_source.observed_by(
    "jwst/nircam/grismr/f444w", samples_per_resolution_element=5)
f322w2_explicit = nircam_source.observed_by(
    "jwst/nircam/grismr/f322w2", wavelength=np.linspace(2.5, 4.0, 300)*u.um)
assert type(f444w_sampled) is Spectrum and type(f322w2_explicit) is Spectrum


## MIRI LRS slitless time series

`jwst/miri/lrs/slitless` selects conventional P750L/SLITLESSPRISM. Its nominal design curve, $R=24\lambda[\mu\mathrm{m}]-80$, has 5–14 µm model support and 5.5–12 µm output. The blue cut leaves convolution padding; the red cut follows commonly useful coverage. Useful observed coverage depends on the source, extraction, and slitless foldover. IP/IPS, fixed slit, WFSS, and MRS are unsupported.

This synthetic example shows smoothing of broad bands and narrow lines. The [instrument reference](../user_guide/instrument_provenance.rst) gives the source and coverage assumptions.

In [ ]:
miri_wave = np.linspace(5.0, 14.0, 45001) * u.um
miri_flux = np.ones(miri_wave.size)
# Broad, shallow synthetic bands plus narrow lines demonstrate smoothing.
for center, depth, sigma in ((6.3, .06, .15), (8.0, .025, .4), (10.5, .2, .001)):
    miri_flux -= depth * np.exp(-.5 * ((miri_wave.value-center)/sigma)**2)
miri_source = Spectrum(spectral_axis=miri_wave, flux=miri_flux * source.flux.unit)
miri_profile = get_profile("jwst/miri/lrs/slitless")
miri = miri_source.observed_by(miri_profile)
miri_sampled = miri_source.observed_by(miri_profile, samples_per_resolution_element=5)
print(miri_profile.mode, miri_profile.quality, miri.valid_intervals)
fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
ax.plot(miri_wave, miri_flux, color="0.7", label="Synthetic input")
ax.plot(miri.wavelength.to_value(u.um), miri.flux.value, label="Native sampling")
ax.plot(miri_sampled.wavelength.to_value(u.um), miri_sampled.flux.value,
        ".", ms=2, label="5 centers/FWHM; integrated bins")
ax.set(xlim=(5.5, 12), xlabel="Wavelength (µm)", ylabel="Relative flux")
ax.legend()
plt.show()


## Scientific limitations

The profiles adopt nominal $\lambda/R$ as Gaussian FWHM, assume negligible intrinsic source widths, and omit non-Gaussian wings and observation-specific corrections. NIRSpec tables supply a nominal 2.2-pixel resolution element from ground/modeling prescriptions. Adopting that width as Gaussian FWHM approximates the wide-slit point-source LSF.

Use the [instrument reference](../user_guide/instrument_provenance.rst) for supported configurations, source citations, and provenance records. The [guide](../user_guide/instrument_profiles.rst) explains padding and source requirements. `profile.provenance` exposes the detailed record offline. WFC3/UVIS G280 remains {ref}`unavailable pending calibration <g280-calibration-status>`.